In [21]:
# Forecast of products demand based on M5 dataset

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from catboost import CatBoostRegressor
import gc


from src.data.load_data import load_data, prepare_calendar
from src.data.build_features import build_features

from src.data.load_data import load_data, prepare_calendar


data_path = "../data/m5"


sales, calendar, prices = load_data(data_path)
calendar = prepare_calendar(calendar)

df = build_features(
    sales=sales,
    calendar=calendar,
    prices=prices,
)
# nonzero_days_14 — число дней с ненулевыми продажами за последние 14 дней.
# nonzero_rate_14 — доля дней с ненулевыми продажами.
# avg_positive_demand_14 — средний объём продаж в дни с положительным спросом.
# adi_14 — средний интервал между днями с ненулевыми продажами.
# demand_regime_14 — классификация спроса на intermittent, regular или unknown.
print(df.shape)
print(df.head())


(58327370, 40)
       item_id  dept_id cat_id store_id state_id    d  sales       date  \
0  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  d_1      3 2011-01-29   
1  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  d_2      0 2011-01-30   
2  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  d_3      0 2011-01-31   
3  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  d_4      1 2011-02-01   
4  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  d_5      4 2011-02-02   

   wm_yr_wk  sell_price  ...  history_days_14  nonzero_rate_14  \
0     11101         2.0  ...              0.0              NaN   
1     11101         2.0  ...              1.0         1.000000   
2     11101         2.0  ...              2.0         0.500000   
3     11101         2.0  ...              3.0         0.333333   
4     11101         2.0  ...              4.0         0.500000   

   avg_positive_demand_14 adi_14 demand_regime_14 price_lag_1 price_median_7  \
0                     NaN    NaN          unknown        

In [3]:
df.columns


Index(['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales',
       'date', 'wm_yr_wk', 'sell_price', 'wday', 'month', 'year',
       'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2',
       'snap_CA', 'snap_TX', 'snap_WI', 'day_of_month', 'week_of_year',
       'lag_1', 'lag_7', 'lag_28', 'rolling_mean_7', 'rolling_mean_14',
       'rolling_mean_28', 'rolling_std_28', 'nonzero_days_14',
       'history_days_14', 'nonzero_rate_14', 'avg_positive_demand_14',
       'adi_14', 'demand_regime_14', 'price_lag_1', 'price_median_7',
       'price_median_28', 'price_ratio_1', 'price_ratio_28'],
      dtype='str')

In [3]:
calendar = prepare_calendar(calendar)

print(calendar.columns.tolist())
print(calendar.head())

['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']
        date  wm_yr_wk    weekday  wday  month  year event_name_1  \
0 2011-01-29     11101   Saturday     1      1  2011          NaN   
1 2011-01-30     11101     Sunday     2      1  2011          NaN   
2 2011-01-31     11101     Monday     3      1  2011          NaN   
3 2011-02-01     11101    Tuesday     4      2  2011          NaN   
4 2011-02-02     11101  Wednesday     5      2  2011          NaN   

  event_type_1 event_name_2 event_type_2  snap_CA  snap_TX  snap_WI  
0          NaN          NaN          NaN        0        0        0  
1          NaN          NaN          NaN        0        0        0  
2          NaN          NaN          NaN        0        0        0  
3          NaN          NaN          NaN        1        1        0  
4          NaN          NaN          NaN        1        0        1  


## We finished with features taken into account. Starting to build seasonal naive baseline

In [5]:
## At first we separate train, validation and test
df_regular =sales3.sort_values(by = ['item_id','store_id','date']).copy()

test_start = df_regular['date'].max() - pd.Timedelta(days=27)
test_df = df_regular[df_regular['date']>=test_start].copy() ## later that test_start
77
train_df = df_regular[df_regular['date']<test_start].copy() ## before test_start

validation_start = test_start - pd.Timedelta(days=27) ## cutoff for validation
validation_df = train_df[train_df['date'] >= validation_start].copy()

train_window = train_df[
    (train_df['date'] >= validation_start - pd.Timedelta(days=91+27)) & (train_df['date']<validation_start)].copy() ## before validation start but half a year period due to limitation of memory

## Here we determine what features are categorical and numerical, clear NaN in categorical features

cat_features = [
    'item_id', 'dept_id','store_id',
    'event_name_1', 'event_type_1', 'demand_regime_14',
]
num_features = [
    'wday','month','year','week_of_year','day_of_month','snap',
    'sell_price','days_after_release',
    'lag1', 'lag7', 'lag28',
    'rolling_mean7', 'rolling_mean28','rolling_median28',
    'rolling_std7','rolling_std28',
    'price1','price7','price28',
    'trend',
    'discount1','discount7','discount28',
    # Intermittent demand features
    'nonzero_days_14',
    'history_days_14',
    'nonzero_rate_14',
    'zero_rate_14',
    'avg_positive_demand_14',
    'adi_14',
    'horizon'
]
feature_cols = cat_features + num_features

del sales1, sales2, sales, sales3
del week_start_sale, week_first_day, start_sale
del calendar, prices

In [26]:

### Baseline - seasonal forecast
def seasonal_naive_forecast(
    train_df,
    horizon=28,
    season_length=7
):
    """
    Seasonal Naive forecast для одной временной серии.
    train_df: DataFrame с колонками date и sales

    horizon: горизонт прогноза в днях.
    season_length: длина сезонности. Для недельной сезонности = 7.
    """
    if train_df['item_id'].nunique()==1 and train_df['store_id'].nunique()==1:
        item_id = train_df['item_id'].unique()[0]
        store_id = train_df['store_id'].unique()[0]
        train_df = train_df.sort_values(by ='date' ).copy()
        diff = (train_df['date'] - train_df['date'].shift(1)).dt.days
        last_date = train_df['date'].iloc[-1]
        future_dates = pd.date_range(start=last_date+pd.Timedelta(days = 1), end=last_date + pd.Timedelta(days=horizon))

        if diff.max(skipna=True)==1 and diff.min(skipna=True)==1:

            df = pd.DataFrame([])
            s = pd.Series([])
            df['forecast_dates'] = future_dates
            df['item_id'] = item_id
            df['store_id'] = store_id
            if season_length < horizon:
                while len(s)< horizon:
                    s = pd.concat([s, train_df.tail(season_length)['sales']])
                df['forecast_sales'] = s[0:horizon].reset_index(drop=True)

            elif season_length >= horizon:
                df['forecast_sales'] = train_df.tail(season_length )[0:horizon]['sales']


        else:
            raise ValueError("additional preprocessing is required for the calendar")
    else:
        raise ValueError('Input train_df must contain exactly one item_id and one store_id')

    return df

all_forecast = []
for (item_id, store_id), group in train_df.groupby(['item_id','store_id']):
    forecast_df = seasonal_naive_forecast(
        train_df=group, horizon=28,
        season_length=7)
    all_forecast.append(forecast_df)
forecasts_df = pd.concat(all_forecast, ignore_index=True)


baseline_forecast = test_df.merge(forecasts_df, how='left', left_on=['item_id','store_id','date'], right_on = ['item_id','store_id','forecast_dates'])
print(baseline_forecast[['item_id','store_id','date','sales','forecast_dates','forecast_sales']].head(10))
if baseline_forecast['forecast_sales'].isna().sum() == 0:
        print('OK with NA')
else: print( 'Please Check NA values in forecast sales')

mae = (baseline_forecast['sales'] - baseline_forecast['forecast_sales']).abs().mean()
print('MAE of baseline = ', round(mae,2))
rmse = (((baseline_forecast['sales'] - baseline_forecast['forecast_sales'])**2).mean())**0.5
print('RMSE of baseline = ', round(rmse,2))

       item_id store_id       date  sales forecast_dates forecast_sales
0  FOODS_1_001     CA_1 2016-03-28    2.0     2016-03-28            0.0
1  FOODS_1_001     CA_1 2016-03-29    1.0     2016-03-29            2.0
2  FOODS_1_001     CA_1 2016-03-30    1.0     2016-03-30            2.0
3  FOODS_1_001     CA_1 2016-03-31    0.0     2016-03-31            0.0
4  FOODS_1_001     CA_1 2016-04-01    4.0     2016-04-01            1.0
5  FOODS_1_001     CA_1 2016-04-02    0.0     2016-04-02            1.0
6  FOODS_1_001     CA_1 2016-04-03    0.0     2016-04-03            0.0
7  FOODS_1_001     CA_1 2016-04-04    4.0     2016-04-04            0.0
8  FOODS_1_001     CA_1 2016-04-05    1.0     2016-04-05            2.0
9  FOODS_1_001     CA_1 2016-04-06    3.0     2016-04-06            2.0
OK with NA
MAE of baseline =  1.23
RMSE of baseline =  2.76


In [27]:
## Now we prepare for the modeling. I'm gonna use single model of CatBoost to predict all horizons. So at first we need to
def make_direct_training_data(df, horizon = 28):
    training_parts = []
    if df['item_id'].nunique()==1 and df['store_id'].nunique()==1:
        for hor in range(1, horizon+1):
            tmp = df.copy()
            tmp['horizon'] = hor
            tmp['target'] = tmp['sales'].shift(-hor)
            tmp['target_date'] = tmp['date'].shift(-hor)
            tmp = tmp.dropna(subset=['target'])
            training_parts.append(tmp)
        training_df = pd.concat(training_parts, ignore_index = True)
    else:
        raise ValueError('Input df must contain exactly one item_id and one store_id')
    return training_df


train_help = []
for (item_id, store_id), group in train_window.groupby(['item_id','store_id']):
    tmp = make_direct_training_data(group, horizon = 28)
    train_help.append(tmp)
training_df = pd.concat(train_help, ignore_index = True)
del train_help

print(training_df.head())
print(training_df['horizon'].value_counts().sort_index())
print(training_df['target'].isna().sum())


       item_id  dept_id cat_id store_id state_id   day  sales       date  \
0  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  1741    0.0 2015-11-04   
1  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  1742    0.0 2015-11-05   
2  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  1743    0.0 2015-11-06   
3  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  1744    1.0 2015-11-07   
4  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  1745    0.0 2015-11-08   

   wday  month  ...  nonzero_days_14 history_days_14 nonzero_rate_14  \
0     5     11  ...              7.0            14.0        0.500000   
1     6     11  ...              6.0            14.0        0.428571   
2     7     11  ...              6.0            14.0        0.428571   
3     1     11  ...              6.0            14.0        0.428571   
4     2     11  ...              7.0            14.0        0.500000   

  zero_rate_14 avg_positive_demand_14    adi_14  demand_regime_14  horizon  \
0     0.500000               1.1

In [28]:
#### We take only nessesary columns from the training dataset. and only 5M of rows due to limitations of memory

X_train = training_df[feature_cols].reset_index(drop = True)
y_train = training_df['target'].reset_index(drop = True)

rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X_train),size=5_000_000,replace=False)

X_train_sample = X_train.iloc[sample_idx].copy()
y_train_sample = y_train.iloc[sample_idx].copy()

# NaN в категориальных признаках
X_train_sample['event_name_1'] = (X_train_sample['event_name_1'].fillna('None').astype(str))

print(X_train_sample[cat_features].isna().sum())


item_id                   0
dept_id                   0
store_id                  0
event_name_1              0
event_type_1        4521214
demand_regime_14          0
dtype: int64


In [9]:
### Here we prepare features for validation

# Origin_date is the last date when we assume that we know sales. It is the last day of train_window
origin_date = training_df['date'].max()
origins = training_df[training_df['date'] == origin_date].copy()


## Features known for the whole validation period which describe state in the future (prices, holidays...)
future_features = validation_df[
    [   'item_id','dept_id','store_id',
        'date','wday','month','year','week_of_year', 'day_of_month',
        'snap',
        'sell_price',
        'event_name_1','event_type_1',
        'days_after_release']].rename(columns={'date': 'target_date'}).copy()
## Features known for every item*store at the moment of origins (1 raw per each item*store )
history_features = origins[
       ['item_id','store_id',
        'lag1','lag7','lag28',
        'rolling_mean7','rolling_mean28','rolling_median28',
        'rolling_std7','rolling_std28',
        'price1','price7','price28',
        'trend',
        'discount1','discount7','discount28']].copy()

###### Now we combine the same structure as the dataframe with ll features, but some of them we know at the origin_date, and others are spread in future with known future characteristics
validation_features = future_features.merge(history_features,how='left',on=['item_id', 'store_id'])
## We determine horizon which is needed for the single CatBoost model
validation_features['horizon'] = (validation_features['target_date'] - origin_date).dt.days
X_validation = validation_features[feature_cols].copy()


for col in cat_features:
    X_validation[col] = X_validation[col].fillna('None')
    X_train_sample[col] = X_train_sample[col].fillna('None')

print('X_train_sample:', X_train_sample.shape)
print('X_validation:', X_validation.shape)

X_train_sample: (5000000, 29)
X_validation: (823230, 29)


In [ ]:
### We prepared everything for building a model, so let's go


from catboost import CatBoostRegressor
import numpy as np
import pandas as pd

# Quantiles we want to predict
quantiles = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]

models = {}
forecasts = {}

for q in quantiles:
    model = CatBoostRegressor(
        loss_function=f'Quantile:alpha={q}',
        eval_metric='MAE',
        iterations=500,
        learning_rate=0.05,
        depth=8,
        random_seed=42,
        verbose=100
    )

    model.fit(
        X_train_sample,
        y_train_sample,
        cat_features=cat_features
    )

    models[q] = model
    forecasts[q] = model.predict(X_validation)

# Combine all quantile forecasts into one dataframe
validation_features = validation_features.copy()

for q in quantiles:
    validation_features[f'forecast_p{int(q * 100)}'] = forecasts[q]

validation_features[
    ['item_id', 'store_id', 'target_date', 'horizon',
     'forecast_p20', 'forecast_p30', 'forecast_p40',
     'forecast_p50', 'forecast_p60', 'forecast_p70',
     'forecast_p80']
].head(10)


### Now we combine the prediction and the actuals
def pinball_loss(y_true, y_pred, q):
    errors = y_true - y_pred
    return np.maximum(q * errors, (q - 1) * errors).mean()


# Join actual sales to forecasts
validation_result = validation_features[
    ['item_id', 'store_id', 'target_date', 'horizon'] +
    [f'forecast_p{int(q * 100)}' for q in quantiles]
].merge(
    validation_df[['item_id', 'store_id', 'date', 'sales']],
    how='left',
    left_on=['item_id', 'store_id', 'target_date'],
    right_on=['item_id', 'store_id', 'date']
).rename(columns={'sales': 'actual'})

# Evaluate each quantile
results = []

for q in quantiles:
    col = f'forecast_p{int(q * 100)}'

    results.append({
        'quantile': f'P{int(q * 100)}',
        'MAE': (validation_result['actual'] -
                validation_result[col]).abs().mean(),
        'Pinball loss': pinball_loss(
            validation_result['actual'],
            validation_result[col],
            q
        ),
        'Coverage': (
            validation_result['actual'] <=
            validation_result[col]
        ).mean()
    })

metrics = pd.DataFrame(results)
print(metrics.round(3))


0:	learn: 1.2800688	total: 1.21s	remaining: 20m 4s
100:	learn: 1.1013311	total: 1m 32s	remaining: 13m 41s
200:	learn: 1.0903246	total: 3m 4s	remaining: 12m 11s
300:	learn: 1.0853511	total: 4m 38s	remaining: 10m 46s
400:	learn: 1.0845253	total: 6m 8s	remaining: 9m 10s
500:	learn: 1.0846295	total: 7m 32s	remaining: 7m 30s
600:	learn: 1.0845144	total: 8m 52s	remaining: 5m 53s
700:	learn: 1.0834405	total: 10m 7s	remaining: 4m 19s
800:	learn: 1.0832693	total: 11m 25s	remaining: 2m 50s
900:	learn: 1.0832246	total: 12m 51s	remaining: 1m 24s
999:	learn: 1.0829021	total: 14m 14s	remaining: 0us
0:	learn: 1.2751220	total: 1.08s	remaining: 17m 55s
100:	learn: 1.0143766	total: 1m 31s	remaining: 13m 37s
200:	learn: 1.0060250	total: 2m 59s	remaining: 11m 52s
300:	learn: 1.0040416	total: 4m 26s	remaining: 10m 19s
400:	learn: 1.0038818	total: 5m 51s	remaining: 8m 45s
500:	learn: 1.0037620	total: 7m 11s	remaining: 7m 9s
600:	learn: 1.0037668	total: 8m 24s	remaining: 5m 35s
700:	learn: 1.0037636	total: 9

In [27]:
### Now we are looking for the best parameter using grid iterations*depth

iterations_list = [500, 1000, 1500]
depth_list = [6, 8, 10]

results = []

for iterations in iterations_list:
    for depth in depth_list:

        print(f'\nTraining: iterations={iterations}, depth={depth}')

        model = CatBoostRegressor(
            loss_function='RMSE',
            eval_metric='MAE',
            iterations=iterations,
            learning_rate=0.05,
            depth=depth,
            random_seed=42,
            verbose=False
        )

        model.fit(
            X_train_sample,
            y_train_sample,
            cat_features=cat_features
        )

        forecast = model.predict(X_validation)

        mae = (
            validation_result['actual'] - forecast
        ).abs().mean()

        rmse = np.sqrt(
            ((validation_result['actual'] - forecast) ** 2).mean()
        )

        results.append({
            'iterations': iterations,
            'depth': depth,
            'MAE': mae,
            'RMSE': rmse
        })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values('MAE')

print(results_df)
# Training: iterations=1500, depth=10
#    iterations  depth       MAE      RMSE
# 3        1000      6  1.063566  2.282704
# 6        1500      6  1.065099  2.288695
# 0         500      6  1.065220  2.282283
# 2         500     10  1.066018  2.292545
# 1         500      8  1.068164  2.284326
# 5        1000     10  1.070801  2.304752
# 7        1500      8  1.073315  2.296634
# 8        1500     10  1.074646  2.314555
# 4        1000      8  1.075338  2.289589


Training: iterations=500, depth=6

Training: iterations=500, depth=8

Training: iterations=500, depth=10

Training: iterations=1000, depth=6

Training: iterations=1000, depth=8

Training: iterations=1000, depth=10

Training: iterations=1500, depth=6

Training: iterations=1500, depth=8

Training: iterations=1500, depth=10
   iterations  depth       MAE      RMSE
3        1000      6  1.063566  2.282704
6        1500      6  1.065099  2.288695
0         500      6  1.065220  2.282283
2         500     10  1.066018  2.292545
1         500      8  1.068164  2.284326
5        1000     10  1.070801  2.304752
7        1500      8  1.073315  2.296634
8        1500     10  1.074646  2.314555
4        1000      8  1.075338  2.289589
